# Project: Production RAG Pipeline

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangChain roadmap.

You will build a production RAG pipeline with query enhancement, retrieval, answer validation, PII protection, conversation memory, and structured citations.

## 1. Install Dependencies

In [ ]:
!pip install -q langchain "langchain[openai]" langgraph langchain-mcp-adapters

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Prepare the Knowledge Base

Create documents and index them into an in-memory vector store.

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(
        page_content="LangChain agents use a ReAct loop: the model reasons about what to do, takes an action using a tool, and observes the result before deciding the next step.",
        metadata={"source": "langchain-docs", "section": "agents"},
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) grounds LLM responses in external data by retrieving relevant documents before generating an answer.",
        metadata={"source": "langchain-docs", "section": "rag"},
    ),
    Document(
        page_content="LangGraph provides a graph-based framework for building stateful, multi-step agent workflows with built-in persistence and streaming.",
        metadata={"source": "langgraph-docs", "section": "overview"},
    ),
    Document(
        page_content="Guardrails protect agents from processing sensitive data. PIIMiddleware detects and redacts emails, credit cards, phone numbers, and SSNs.",
        metadata={"source": "langchain-docs", "section": "guardrails"},
    ),
    Document(
        page_content="Structured output uses Pydantic models with response_format to ensure the agent returns data in a predictable schema.",
        metadata={"source": "langchain-docs", "section": "structured-output"},
    ),
    Document(
        page_content="Memory in LangGraph agents uses checkpointers like InMemorySaver to persist conversation state across multiple turns.",
        metadata={"source": "langgraph-docs", "section": "memory"},
    ),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(documents)} documents. Retriever ready.")

## 4. Query Enhancement Tool

Enhance user queries with domain-specific terms for better retrieval.

In [ ]:
from langchain_core.tools import tool

@tool
def enhance_query(query: str) -> str:
    """Enhance a user query for better retrieval results."""
    expansions = {
        "agent": "LangChain agent ReAct loop reasoning tools",
        "rag": "RAG retrieval augmented generation documents embeddings",
        "memory": "memory persistence conversation state checkpointer saver",
        "guardrails": "guardrails PII safety middleware protection",
    }
    enhanced = query
    for keyword, expansion in expansions.items():
        if keyword.lower() in query.lower():
            enhanced = f"{query} {expansion}"
            break
    return enhanced

print(f"Original: 'How do agents work?'")
print(f"Enhanced: '{enhance_query.invoke('How do agents work?')}'")

## 5. Retriever Tool

Wrap the retriever in a tool with source citations.

In [ ]:
@tool
def search_docs(query: str) -> str:
    """Search the documentation knowledge base."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found."

    results = []
    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        section = doc.metadata.get("section", "general")
        results.append(f"[{source} > {section}]\n{doc.page_content}")
    return "\n\n---\n\n".join(results)

print(search_docs.invoke("How do agents work?"))

## 6. Answer Validation Guardrail

Middleware that checks answers are grounded in source documents.

In [ ]:
from langgraph.prebuilt.middleware import AgentMiddleware

class AnswerValidationMiddleware(AgentMiddleware):
    def __init__(self, required_phrases=None):
        self.required_phrases = required_phrases or ["source", "according to", "based on"]

    def after_agent(self, state):
        response = state["messages"][-1].content.lower()
        has_citation = any(phrase in response for phrase in self.required_phrases)
        if not has_citation and len(response) > 100:
            state["messages"][-1].content += (
                "\n\n_Note: This response may not be fully grounded in source documents._"
            )
        return state

print("AnswerValidationMiddleware defined.")

## 7. PII Middleware

Add PII protection to redact sensitive data from inputs.

In [ ]:
from langgraph.prebuilt.middleware import PIIMiddleware

pii_middleware = PIIMiddleware(strategy="redact")
print("PIIMiddleware configured with 'redact' strategy.")

## 8. Conversation Memory

Use `InMemorySaver` to persist conversation state across turns.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()
print("InMemorySaver checkpointer ready.")

## 9. Structured Output — CitedAnswer

Define a schema that ensures responses include citations and confidence.

In [ ]:
from pydantic import BaseModel, Field

class CitedAnswer(BaseModel):
    answer: str = Field(description="The answer to the user's question")
    citations: list[str] = Field(description="List of source documents cited")
    confidence: str = Field(description="Confidence level: high, medium, or low")
    follow_up_questions: list[str] = Field(description="Suggested follow-up questions")

print("CitedAnswer schema fields:", list(CitedAnswer.model_fields.keys()))

## 10. Assemble the Production Agent

Combine all components — tools, middleware, memory, and structured output.

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4o-mini", model_provider="openai")

validation_middleware = AnswerValidationMiddleware()

agent = create_react_agent(
    model=model,
    tools=[enhance_query, search_docs],
    prompt=(
        "You are a documentation assistant. For every question:\n"
        "1. Use enhance_query to improve the search terms\n"
        "2. Use search_docs to find relevant documentation\n"
        "3. Answer based ONLY on retrieved documents\n"
        "4. Always cite your sources with [source > section] format\n"
        "If no relevant documents are found, say so clearly."
    ),
    checkpointer=memory,
    middleware=[pii_middleware, validation_middleware],
    response_format=CitedAnswer,
)

print("Production RAG agent assembled.")

## 11. Query the Pipeline

Ask questions and get cited, structured answers.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "user-session-1"}}

result = agent.invoke(
    {"messages": [HumanMessage(content="How do LangChain agents work?")]},
    config=config,
)
print(result["messages"][-1].content)

## 12. Conversation Memory in Action

Follow up in the same conversation — the agent remembers context.

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="How do I add memory to them?")]},
    config=config,
)
print(result["messages"][-1].content)

## 13. PII Redaction in Action

Test that sensitive data is redacted before the agent processes it.

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="My email is alice@example.com, tell me about guardrails")]},
    config=config,
)
print(result["messages"][-1].content)

## 14. Multiple Queries

Run several queries through the production pipeline.

In [ ]:
new_config = {"configurable": {"thread_id": "user-session-2"}}

questions = [
    "What is RAG and how does it work?",
    "How do I add structured output to my agent?",
    "What guardrails are available?",
]

for question in questions:
    result = agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=new_config,
    )
    print(f"Q: {question}")
    print(f"A: {result['messages'][-1].content[:200]}...\n")

## Summary

- Production RAG combines query enhancement, retrieval, validation, PII protection, memory, and structured output
- `enhance_query` expands user queries with domain terms for better retrieval
- `AnswerValidationMiddleware` checks that responses cite source documents
- `PIIMiddleware` redacts sensitive data before agent processing
- `InMemorySaver` maintains conversation context via `thread_id`
- `CitedAnswer` schema ensures responses include citations, confidence, and follow-ups